# 10 — Summarisation, QA, RAG, ROUGE, and faithfulness checks

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Summarisation compresses information. QA answers questions from context. RAG retrieves evidence before generation.

```text
question → retrieve evidence → generate answer grounded in evidence
```

This notebook implements extractive summarisation, TF-IDF retrieval, a tiny RAG skeleton, and ROUGE-L evaluation from scratch.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "The product dashboard now loads in under two seconds. Customers praised the faster analytics experience.",
    "Several users reported that PDF invoice uploads crash the app. The bug affects large files over 20MB.",
    "Billing complaints increased after some customers were charged twice for subscriptions. Support issued refunds.",
    "The AI summary feature sometimes invented numbers that were not present in uploaded reports. A citation feature is planned.",
    "Enterprise customers requested SSO and role-based access controls. SSO has been released, while RBAC is in development.",
]

## Extractive summarisation with TextRank-style sentence graph

Sentences are nodes. Similarity creates edges. Important sentences are similar to other important sentences.

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def textrank_summary(text, top_k=2, iterations=30, damping=0.85):
    sentences = split_sentences(text)
    if len(sentences) <= top_k:
        return sentences
    X = TfidfVectorizer().fit_transform(sentences)
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    # Normalize rows
    row_sums = sim.sum(axis=1, keepdims=True) + 1e-9
    M = sim / row_sums
    scores = np.ones(len(sentences)) / len(sentences)
    for _ in range(iterations):
        scores = (1 - damping) / len(sentences) + damping * M.T.dot(scores)
    top = sorted(np.argsort(scores)[-top_k:])
    return [sentences[i] for i in top]

long_text = " ".join(documents)
textrank_summary(long_text, top_k=3)

## TF-IDF retrieval

This is a simple retrieval baseline. For production RAG, replace it with embeddings/vector search or hybrid search.

In [ ]:
class TfidfRetriever:
    def __init__(self, docs):
        self.docs = docs
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
        self.X = self.vectorizer.fit_transform(docs)

    def search(self, query, top_k=3):
        q = self.vectorizer.transform([query])
        sims = cosine_similarity(q, self.X)[0]
        order = np.argsort(sims)[::-1][:top_k]
        return [{"doc_id": int(i), "score": float(sims[i]), "text": self.docs[i]} for i in order]

retriever = TfidfRetriever(documents)
retriever.search("Which issue caused hallucinated numbers in reports?", top_k=2)

## Tiny RAG skeleton

In a real app, replace `generate_answer_stub` with your LLM call. Keep retrieval, prompt construction, citations, and validation outside the model.

In [ ]:
def build_rag_prompt(question, retrieved):
    evidence = "\n".join([f"[{r['doc_id']}] {r['text']}" for r in retrieved])
    return f"""Answer the question using only the evidence below. Cite document ids in square brackets.

Evidence:
{evidence}

Question: {question}
Answer:"""

def generate_answer_stub(question, retrieved):
    # Simple extractive fallback: return the highest-scoring chunk.
    best = retrieved[0]
    return f"Based on document [{best['doc_id']}]: {best['text']}"

question = "What are the main risks in the AI summary feature?"
retrieved = retriever.search(question, top_k=3)
prompt = build_rag_prompt(question, retrieved)
answer = generate_answer_stub(question, retrieved)
print(prompt)
print("\n---ANSWER---")
print(answer)

## Extractive QA baseline

A simple baseline selects the sentence with highest overlap/retrieval score. This is not a replacement for BERT QA, but it is a useful product fallback.

In [ ]:
def answer_extractive(question, docs):
    sentences = []
    for doc_id, doc in enumerate(docs):
        for sent in split_sentences(doc):
            sentences.append((doc_id, sent))
    retr = TfidfRetriever([s for _, s in sentences])
    hit = retr.search(question, top_k=1)[0]
    doc_id, sent = sentences[hit["doc_id"]]
    return {"answer": sent, "source_doc": doc_id, "score": hit["score"]}

answer_extractive("What did users report about PDF invoice uploads?", documents)

## ROUGE-L from scratch

ROUGE-L uses longest common subsequence. It is often used for summarisation evaluation, but it measures overlap, not factual faithfulness.

In [ ]:
def lcs_len(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

def rouge_l(candidate, reference):
    cand = re.findall(r"\w+", candidate.lower())
    ref = re.findall(r"\w+", reference.lower())
    lcs = lcs_len(cand, ref)
    prec = lcs / max(len(cand), 1)
    rec = lcs / max(len(ref), 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    return {"rouge_l_precision": prec, "rouge_l_recall": rec, "rouge_l_f1": f1}

candidate = "PDF invoice uploads crash for large files. Billing refunds were issued."
reference = "Users reported PDF invoice upload crashes for files over 20MB, and support issued refunds for billing complaints."
rouge_l(candidate, reference)

## Faithfulness checks

For high-stakes products, overlap metrics are not enough. Add checks:

- every answer sentence has supporting evidence
- numbers/dates/entities in answer appear in retrieved context
- answer says "not found" when evidence is missing
- citations point to the right chunk

In [ ]:
def extract_numbers(text):
    return re.findall(r"\d+(?:\.\d+)?", text)

def numbers_supported(answer, evidence_texts):
    evidence_numbers = set(n for e in evidence_texts for n in extract_numbers(e))
    return all(n in evidence_numbers for n in extract_numbers(answer))

answer_good = "The upload bug affects files over 20MB."
answer_bad = "The upload bug affects files over 50MB."
evidence = [documents[1]]
print(numbers_supported(answer_good, evidence))
print(numbers_supported(answer_bad, evidence))

## Product RAG checklist

- chunk documents with stable ids
- store metadata and permissions
- retrieve with hybrid lexical + semantic search
- rerank if needed
- build prompts that require citations
- validate output format and factual claims
- log retrieved sources, answer, user feedback, and failure mode